<a href="https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ubaid8878/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
My baseline rule prioritizes pages that are stale and have weaker CTR. A page is considered stale when it has not been updated for 180 days or more. A page has low CTR when its CTR is below the median CTR of the dataset.

Reason codes:
- STALE_LOW_CTR: page is stale and has low CTR.
- STALE: page is stale but does not have low CTR.
- LOW_CTR: page has low CTR but is not stale.
- NO_FLAG: page is neither stale nor low CTR.

In [3]:
import os

print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

Current folder: /content
Files here: ['.config', 'sample_data']


In [4]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/ubaid8878/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

# Clone your GitHub repository
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Move into your repository
os.chdir(REPO_DIR)

# Check location
print("Current folder:", os.getcwd())
print("Top-level files:", os.listdir())

Current folder: /content/Flyrank-ML-Internship
Top-level files: ['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs', '.git', 'AGENTS.md', 'LICENSE', '.github', 'submission', 'CLAUDE.md', 'GUIDE.md', '.gitignore', 'requirements.txt']


In [5]:
import os

data_path = "data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(data_path))

if os.path.exists(data_path):
    print("Dataset found:", data_path)
else:
    print("Dataset NOT found")

Dataset exists: True
Dataset found: data/raw/content_refresh_anonymized.csv


In [7]:
import pandas as pd
import numpy as np

# Load the dataset into df
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully!
Rows: 30000
Columns: 44


In [8]:
import numpy as np

# Define the two signals
stale = df["days_since_last_update"] >= 180
low_ctr = df["ctr"] < df["ctr"].median()

# Create one reason code for every page
df["reason_code"] = np.select(
    [
        stale & low_ctr,
        stale,
        low_ctr
    ],
    [
        "STALE_LOW_CTR",
        "STALE",
        "LOW_CTR"
    ],
    default="NO_FLAG"
)

print("CTR median:", round(df["ctr"].median(), 4))

print("\nReason code counts:")
print(df["reason_code"].value_counts())

CTR median: 0.07

Reason code counts:
reason_code
NO_FLAG          15136
LOW_CTR          14690
STALE_LOW_CTR      120
STALE               54
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
The rule gives the highest priority to pages that are both stale and have low CTR. These pages receive the highest score because they have two signals that suggest they may need attention. Stale pages receive a lower score, and low-CTR pages receive a smaller score. The queue is ranked from highest to lowest score.


In [9]:
# Section 2: Build the ranked action queue

df["baseline_score"] = (
    (df["reason_code"] == "STALE_LOW_CTR").astype(int) * 3
    + (df["reason_code"] == "STALE").astype(int) * 2
    + (df["reason_code"] == "LOW_CTR").astype(int) * 1
)

df["action"] = np.select(
    [
        df["reason_code"] == "STALE_LOW_CTR",
        df["reason_code"] == "STALE",
        df["reason_code"] == "LOW_CTR"
    ],
    [
        "REVIEW_REFRESH_AND_CTR",
        "REVIEW_REFRESH",
        "REVIEW_CTR"
    ],
    default="NO_ACTION"
)

queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

# Save the ranked queue
import os
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Saved:", output_path)
print("\nTop 20:")
print(
    queue[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "ctr",
            "impressions_90d"
        ]
    ].head(20).to_string(index=False)
)

Saved: work/outputs/baseline_action_score.csv

Top 20:
          content_id  baseline_score   reason_code                 action  days_since_last_update  ctr  impressions_90d
content_5feee3994adb               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.01             7812
content_b16bd7307b39               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.00             4590
content_074ba6ead17b               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              533
content_fd16e3475c29               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              429
content_6476d1d8c050               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     313 0.00              304
content_4f241bad48a3               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     236 0.00              285
content_ea41fe5cf292               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# Section 3: display the top 20 for review

top20 = queue.head(20).copy()

review_columns = [
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "impressions_90d"
]

print(top20[review_columns].to_string(index=False))


          content_id  baseline_score   reason_code                 action  days_since_last_update  ctr  impressions_90d
content_5feee3994adb               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.01             7812
content_b16bd7307b39               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     194 0.00             4590
content_074ba6ead17b               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              533
content_fd16e3475c29               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              429
content_6476d1d8c050               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     313 0.00              304
content_4f241bad48a3               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     236 0.00              285
content_ea41fe5cf292               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183 0.00              265
content_d25a099b3726               3 STA

In [ ]:
Top-20 review:

All top-20 pages received the same baseline score of 3 because they meet both rule conditions: they are at least 180 days since their last update and their CTR is below the dataset median of 0.07. The action for all of them is REVIEW_REFRESH_AND_CTR.

1. content_5feee3994adb — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 194 days and CTR is 0.01. Confidence: relatively high because it has 7,812 impressions. What would make it wrong: the page may already have a valid reason for low CTR or may not benefit from a refresh.

2. content_b16bd7307b39 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 194 days and CTR is 0.00. Confidence: relatively high because it has 4,590 impressions. What would make it wrong: low CTR may be explained by its position or content type rather than staleness.

3. content_074ba6ead17b — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: moderate because it has 533 impressions. What would make it wrong: the page may have too little traffic for the CTR signal to be reliable.

4. content_fd16e3475c29 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: moderate because it has 429 impressions. What would make it wrong: the low CTR could be caused by limited impressions or another page-level factor.

5. content_6476d1d8c050 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 313 days and CTR is 0.00. Confidence: moderate because it has 304 impressions. What would make it wrong: the page may not have enough traffic for the CTR signal to be dependable.

6. content_4f241bad48a3 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 236 days and CTR is 0.00. Confidence: moderate because it has 285 impressions. What would make it wrong: low CTR may not necessarily mean that refreshing the page will improve performance.

7. content_ea41fe5cf292 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: moderate because it has 265 impressions. What would make it wrong: the low CTR may be related to factors other than page freshness.

8. content_d25a099b3726 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 305 days and CTR is 0.00. Confidence: moderate because it has 202 impressions. What would make it wrong: the page may have limited evidence of a meaningful CTR problem.

9. content_958a46db26bd — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: moderate because it has 198 impressions. What would make it wrong: the low impression volume may make the CTR less stable.

10. content_02b0d6e30129 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 313 days and CTR is 0.00. Confidence: moderate because it has 176 impressions. What would make it wrong: the page may have insufficient traffic to justify a refresh.

11. content_f488400fca67 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 305 days and CTR is 0.00. Confidence: moderate because it has 155 impressions. What would make it wrong: the small number of impressions makes the CTR signal less reliable.

12. content_b6e4581523ed — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: moderate because it has 104 impressions. What would make it wrong: the CTR may be unstable because of the relatively small number of impressions.

13. content_ab27c30d81f4 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 304 days and CTR is 0.00. Confidence: low-to-moderate because it has only 103 impressions. What would make it wrong: the sample may be too small to make a strong decision.

14. content_7a888d3d99c8 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 313 days and CTR is 0.00. Confidence: low because it has only 95 impressions. What would make it wrong: the low impression count makes the CTR signal weak.

15. content_07ce98c6085a — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 304 days and CTR is 0.00. Confidence: low because it has only 85 impressions. What would make it wrong: the page may simply have too little traffic to support the decision.

16. content_30eb41dff556 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: low because it has only 84 impressions. What would make it wrong: the CTR estimate may be unreliable at this traffic level.

17. content_2e2a634851a0 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: low because it has only 81 impressions. What would make it wrong: the low impression count may not provide enough evidence for action.

18. content_460b11dcac6a — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: low because it has only 81 impressions. What would make it wrong: the CTR may change substantially with more impressions.

19. content_d34c89fad803 — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: low because it has only 80 impressions. What would make it wrong: the small sample makes the CTR signal uncertain.

20. content_3a93f78aa0a — Action: REVIEW_REFRESH_AND_CTR. Reason: stale by 183 days and CTR is 0.00. Confidence: low because it has only 79 impressions. What would make it wrong: the low volume makes it difficult to confidently treat the CTR as a problem.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# Section 4: Find weak picks

weak_picks = queue[
    (queue["baseline_score"] == 3) &
    (queue["impressions_90d"] < 100)
].head(10)

print("Weak picks:")
print(
    weak_picks[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "days_since_last_update",
            "ctr",
            "impressions_90d"
        ]
    ].to_string(index=False)
)


Weak picks:
          content_id  baseline_score   reason_code                 action  days_since_last_update  ctr  impressions_90d
content_7a888d3d99c8               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     313  0.0               95
content_07ce98c6085a               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     304  0.0               85
content_30eb41dff556               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183  0.0               84
content_2e2a634851a0               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183  0.0               81
content_460b11dcac6a               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183  0.0               81
content_d34c89fad803               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183  0.0               80
content_3a93f78aa0a5               3 STALE_LOW_CTR REVIEW_REFRESH_AND_CTR                     183  0.0               79
content_bbca724138f2        

In [ ]:
Weak picks:

Some of the lower-ranked top-20 pages are weak picks because they have very low impression counts. For example, several pages have fewer than 100 impressions while also having a CTR of 0.00. The rule therefore flags them because they are stale and have low CTR, but the small number of impressions makes the CTR signal less reliable.

This is a limitation of the baseline rule. A stronger system could account for impression volume or statistical uncertainty before recommending an action.

Leakage check:

The baseline rule uses only observable pre-decision signals: days_since_last_update, ctr, and impressions_90d. It does not use trend_direction, trend_pct, a future outcome, or product decision flags. Therefore, no label-derived or future-window information was intentionally used in the rule.

In [12]:
# Explicit leakage check

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Leakage columns present in dataset:")
for col in leakage_columns:
    print(col, "->", col in df.columns)

print("\nFeatures used by my baseline rule:")
print([
    "days_since_last_update",
    "ctr",
    "impressions_90d"
])

Leakage columns present in dataset:
trend_direction -> True
trend_pct -> True
is_declining_label -> False

Features used by my baseline rule:
['days_since_last_update', 'ctr', 'impressions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.